# Hydra nixpkgs exploration

This notebook is self-contained. It talks to Hydra directly and does not import any local project code.

In [ ]:
from __future__ import annotations

import json
import urllib.error
import urllib.request
from dataclasses import dataclass
from datetime import UTC, datetime


@dataclass(frozen=True)
class JobBuildInfo:
    build_id: int
    job: str
    jobset: str
    project: str
    system: str
    nix_name: str
    finished_at: str | None
    eval_ids: tuple[int, ...]


def fetch_json(url: str) -> dict:
    request = urllib.request.Request(
        url,
        headers={
            "Accept": "application/json",
            "User-Agent": "hydra-notebook",
        },
    )
    try:
        with urllib.request.urlopen(request) as response:
            return json.load(response)
    except urllib.error.HTTPError as error:
        message = error.read().decode("utf-8", errors="replace").strip()
        detail = f": {message}" if message else ""
        raise RuntimeError(f"{url} returned HTTP {error.code}{detail}") from error
    except urllib.error.URLError as error:
        raise RuntimeError(f"failed to fetch {url}: {error.reason}") from error


def iso8601_utc(timestamp: int | None) -> str | None:
    if timestamp is None:
        return None
    return datetime.fromtimestamp(timestamp, UTC).isoformat().replace("+00:00", "Z")


def latest_build_url(base_url: str, project: str, jobset: str, job: str) -> str:
    return f"{base_url.rstrip('/')}/job/{project}/{jobset}/{job}/latest-finished"


def eval_url(base_url: str, eval_id: int) -> str:
    return f"{base_url.rstrip('/')}/eval/{eval_id}"


def hydra_job_name(package: str, system: str | None) -> str:
    return f"{package}.{system}" if system else package


def fetch_latest_job_build(*, base_url: str, project: str, jobset: str, job: str) -> JobBuildInfo:
    build = fetch_json(latest_build_url(base_url, project, jobset, job))
    if build.get("finished") != 1:
        raise RuntimeError(f"latest finished build for {job} is not marked finished")
    if build.get("buildstatus") != 0:
        raise RuntimeError(
            f"latest finished build for {job} did not succeed "
            f"(buildstatus={build.get('buildstatus')})"
        )

    eval_ids = build.get("jobsetevals") or []
    return JobBuildInfo(
        build_id=int(build["id"]),
        job=str(build["job"]),
        jobset=str(build["jobset"]),
        project=str(build["project"]),
        system=str(build["system"]),
        nix_name=str(build.get("nixname") or ""),
        finished_at=iso8601_utc(build.get("stoptime")),
        eval_ids=tuple(sorted(int(eval_id) for eval_id in eval_ids)),
    )


def filter_evals(info: JobBuildInfo, max_eval_id: int | None) -> JobBuildInfo:
    eval_ids = tuple(eval_id for eval_id in info.eval_ids if max_eval_id is None or eval_id <= max_eval_id)
    return JobBuildInfo(
        build_id=info.build_id,
        job=info.job,
        jobset=info.jobset,
        project=info.project,
        system=info.system,
        nix_name=info.nix_name,
        finished_at=info.finished_at,
        eval_ids=eval_ids,
    )


def common_eval_ids(job_infos: list[JobBuildInfo]) -> set[int]:
    if not job_infos:
        return set()
    common = set(job_infos[0].eval_ids)
    for info in job_infos[1:]:
        common &= set(info.eval_ids)
    return common


def fetch_revision_for_eval(base_url: str, eval_id: int, input_name: str) -> str:
    evaluation = fetch_json(eval_url(base_url, eval_id))
    inputs = evaluation.get("jobsetevalinputs") or {}
    input_info = inputs.get(input_name)
    if not input_info:
        available = ", ".join(sorted(inputs)) or "none"
        raise RuntimeError(
            f"evaluation {eval_id} does not have input {input_name!r}; available inputs: {available}"
        )
    revision = input_info.get("revision")
    if not revision:
        raise RuntimeError(f"evaluation {eval_id} input {input_name!r} has no revision")
    return str(revision)


In [ ]:
BASE_URL = "https://hydra.nixos.org"
PROJECT = "nixpkgs"
JOBSET = "trunk"
INPUT_NAME = "nixpkgs"
MAX_EVAL_ID = None

JOBS = [
    hydra_job_name("hello", "x86_64-linux"),
    hydra_job_name("hello", "aarch64-darwin"),
]

BASE_URL, PROJECT, JOBSET, INPUT_NAME, MAX_EVAL_ID, JOBS

In [ ]:
job_infos = []

for job in JOBS:
    info = fetch_latest_job_build(
        base_url=BASE_URL,
        project=PROJECT,
        jobset=JOBSET,
        job=job,
    )
    info = filter_evals(info, MAX_EVAL_ID)
    job_infos.append(info)

    print(job)
    print(f"  build_id:    {info.build_id}")
    print(f"  system:      {info.system}")
    print(f"  finished_at: {info.finished_at}")
    print(f"  eval_count:  {len(info.eval_ids)}")
    print(f"  newest eval: {max(info.eval_ids) if info.eval_ids else None}")
    print()


In [ ]:
overlap = common_eval_ids(job_infos)
latest_common_eval = max(overlap) if overlap else None

print(f"common eval count: {len(overlap)}")
print(f"latest common eval: {latest_common_eval}")

In [ ]:
if latest_common_eval is None:
    print("No common eval across the selected jobs.")
else:
    revision = fetch_revision_for_eval(BASE_URL, latest_common_eval, INPUT_NAME)
    print(f"eval_id:   {latest_common_eval}")
    print(f"revision:  {revision}")

In [ ]:
# Optional: inspect the newest few eval ids for each job.
{info.job: info.eval_ids[-10:] for info in job_infos}